# 34. Weak references and object lifetime

**Level:** Advanced

**Learning outcome:** Distinguish observation from ownership and avoid using garbage collection as a resource protocol.

**Prerequisites:** 04, 13, and 21: ownership, deterministic cleanup, and slots.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](33_Async_Context_Managers_Iteration_and_Cancellation.ipynb) | [Next notebook](35_SOLID_Refactoring_and_Dispatch.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
A strong reference keeps an object reachable. A weak reference can observe a supported object
without keeping it alive. Calling a weakref returns the object while it exists, or None afterward.
Save the returned object in a local variable if you need to use it; checking one weakref call and
then using a second call can race with the last strong reference disappearing.

WeakValueDictionary is useful for a non-owning index or optional cache. It does not promise
retention: an entry can disappear when no strong owner remains. A real inventory repository
normally owns its data and should not accidentally drop devices because it used weak values.
WeakMethod can avoid a long-lived event publisher keeping an entire subscriber alive through a
bound method.

Garbage collection timing is not a cross-implementation resource-lifetime guarantee. These
deterministic examples call gc.collect to demonstrate reachability after removing known owners.
Do not use __del__, finalizers, or weakref callbacks as your primary network session close policy.
Use with/async with and explicit application ownership.

Some types do not support weak references. A slotted class can allow them by including __weakref__
when its hierarchy does not already provide that support.

## Worked example 1: A weak reference observes but does not own

**Problem:** Show the difference between a retained object and a non-owning reference.

### Step-by-step explanation
1. Construct one user-defined instance and a weak reference.
2. Temporarily retrieve the instance, which creates another strong reference.
3. Delete every known strong owner before requesting a collection.
4. Verify the weak reference returns None rather than an apparently usable object.

**Expected output:**
```text
weak reference no longer resolves
```

**Common mistake / limitation:** Accidentally leaving the observed local alive would keep the object reachable and invalidate this demonstration.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
import gc
import weakref

class Device:
    pass

device = Device()
reference = weakref.ref(device)
observed = reference()
assert observed is device
del observed
del device
gc.collect()
assert reference() is None
print("weak reference no longer resolves")

weak reference no longer resolves


### Practice 1

Add a second strong owner and show why deleting only the first owner does not clear the weak reference.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: A weak value index is not a durable repository

**Problem:** Demonstrate entry removal when the real owner disappears.

### Step-by-step explanation
1. Create a weak-value mapping, not a dictionary with strong values.
2. Keep the device alive through its explicit owner.
3. Remove that owner and collect unreachable objects.
4. Check membership again to show the cache does not retain the device.

**Expected output:**
```text
non-owning index entry removed
```

**Common mistake / limitation:** Do not use this as authoritative persistent inventory storage.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
import gc
from weakref import WeakValueDictionary

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

index = WeakValueDictionary()
device = Device("edge-01")
index[device.hostname] = device
assert index["edge-01"] is device
del device
gc.collect()
assert "edge-01" not in index
print("non-owning index entry removed")

non-owning index entry removed


### Practice 2

Keep a strong repository beside the weak index and compare their ownership roles.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: A slotted class explicitly supports weak references

**Problem:** Combine deliberate storage with a non-owning observation path.

### Step-by-step explanation
1. Include __weakref__ in the direct slotted class's storage declaration.
2. Confirm instances still have no ordinary attribute dictionary.
3. Create and resolve a weak reference while an owner exists.
4. Delete the owner and verify observation ends.

**Expected output:**
```text
slotted weak-reference support verified
```

**Common mistake / limitation:** Slots, weak references, and inheritance interact; do not add a duplicate __weakref__ slot if a base already provides it.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
import gc
import weakref

class Device:
    __slots__ = ("hostname", "__weakref__")

    def __init__(self, hostname):
        self.hostname = hostname

device = Device("edge-01")
reference = weakref.ref(device)
assert reference() is device
assert not hasattr(device, "__dict__")
del device
gc.collect()
assert reference() is None
print("slotted weak-reference support verified")

slotted weak-reference support verified


### Practice 3

Create a slotted class without __weakref__ and assert the specific weakref construction failure.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Weak references and object lifetime

**You should now be able to:** Distinguish observation from ownership and avoid using garbage collection as a resource protocol.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/library/weakref.html`
- `https://docs.python.org/3.13/reference/datamodel.html`
- `https://docs.python.org/3.13/library/gc.html`

References describe Python contracts. They do not certify any vendor command or network policy.